# 🧠 Precision and Bounding Box Thresholding

Welcome to the hands-on explanation notebook for **Precision**! In this notebook, we will:
1. Define the mathematical formulation of Precision.
2. Implement Precision from scratch and verify it against `scikit-learn`.
3. Simulate a drone wellhead inspection dataset to explore how positive predictions behave under different confidence thresholds.
4. Visualize the **Precision-Threshold Curve** to see how increasing model confidence filters out False Positives (alarms).
5. Explain how setting thresholds in YOLO affects industrial deployment costs.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_score

# Set seed for reproducibility
np.random.seed(42)

## 1. Calculating Precision from Scratch

Let's write a function to calculate Precision:
$$\text{Precision} = \frac{TP}{TP + FP}$$

In [ ]:
def calculate_precision(y_true, y_pred):
    """
    Calculate Precision score from scratch.
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    
    total_pred_positive = TP + FP
    if total_pred_positive == 0:
        return 1.0
    return TP / total_pred_positive

# Test arrays (e.g., bounding box classification checks)
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

prec_scratch = calculate_precision(y_true, y_pred)
prec_sklearn = precision_score(y_true, y_pred)

print(f"Custom Precision : {prec_scratch:.4f}")
print(f"Sklearn Precision: {prec_sklearn:.4f}")

## 2. Exploring Bounding Box Confidence and Precision

In object detection, each bounding box has a predicted probability score (confidence). 
We choose a **Confidence Threshold** to decide if a box is kept or filtered out:
-   If a box confidence is $\ge \text{threshold}$, we predict Class 1.
-   If it is $< \text{threshold}$, we predict Class 0 (background).

Let's generate 200 bounding boxes with true labels (wellhead vs. background) and their model confidence scores, and analyze how varying the threshold affects the model's Precision.

In [ ]:
# Generate 200 samples
n_samples = 200

# 30 actual wellheads (Class 1), 170 actual background/false anomalies (Class 0)
y_true_wellheads = np.concatenate([np.ones(30), np.zeros(170)]).astype(int)

# Generate confidence scores:
# Actual wellheads have high confidence (mean 0.8)
conf_wellheads = np.random.normal(0.8, 0.15, 30)
# Background clutter has low confidence (mean 0.3)
conf_bg = np.random.normal(0.3, 0.18, 170)

# Combine and clip confidence scores
confidence_scores = np.concatenate([conf_wellheads, conf_bg])
confidence_scores = np.clip(confidence_scores, 0.0, 1.0)

Now let's compute Precision across a range of confidence thresholds from `0.0` to `0.95`.

In [ ]:
thresholds = np.linspace(0.0, 0.95, 100)
precision_history = []
detections_kept = []

for threshold in thresholds:
    y_pred_temp = (confidence_scores >= threshold).astype(int)
    prec = calculate_precision(y_true_wellheads, y_pred_temp)
    precision_history.append(prec)
    detections_kept.append(np.sum(y_pred_temp == 1))

# Plot the Precision-Threshold Curve
plt.figure(figsize=(10, 5))
plt.plot(thresholds, precision_history, color='teal', linewidth=3, label='Precision')
plt.xlabel('Confidence Threshold')
plt.ylabel('Precision Score')
plt.title('Precision vs. Confidence Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.axvline(0.5, color='red', linestyle='--', alpha=0.7, label='Standard Default (0.50)')
plt.legend()
plt.show()

Observe that:
-   For low thresholds (e.g. `0.2`), Precision is very low ($\approx 25\%$), meaning that $75\%$ of the alarms sent to the operators are false alarms!
-   For high thresholds (e.g. `0.75`), Precision goes up to **$100\%$**, meaning that every single wellhead alert sent is correct, eliminating false maintenance trips.

Let's look at the tradeoff: how many wellheads did we actually find?

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(thresholds, detections_kept, color='purple', linewidth=2.5, label='Number of Bounding Boxes Kept')
plt.xlabel('Confidence Threshold')
plt.ylabel('Count')
plt.title('Total Positive Detections Kept vs. Threshold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

As the threshold rises, the total number of predicted positives falls. At very high thresholds, we may miss some real wellheads (lower Recall), showing the fundamental **Precision-Recall Tradeoff**.

## 💡 Connection to Computer Vision & YOLO
*   **Confidence Filtering (`conf` argument):** When running YOLO inference, you can control the precision using the `conf` parameter:
    ```bash
    yolo predict model=yolo26n.pt source=drone.mp4 conf=0.75
    ```
    If your priority is eliminating false alarms (e.g., inspecting a wellhead that is far away), setting `conf=0.75` forces the model to achieve very high precision.